<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/04_FinBERT_categorical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinBERT for Predicting News Sentiment

## Install Libraries

In [1]:
# ! pip install contractions emoji gensim optuna torch matplotlib

## Iport Libraries

In [ ]:
# Common Python Libraries
import numpy as np
import pandas as pd
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import random
from datasets import Dataset

# Deep Learning Libraries
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback
from torch.optim import Adam, AdamW
from accelerate import Accelerator

# Data Preprocessing
from sklearn.model_selection import train_test_split

## Download nltk dependencies
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

# Model metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, RocCurveDisplay, f1_score, precision_score, recall_score, accuracy_score

# Model Tracking
import wandb

# Google Colab Setup
# from google.colab import drive
# drive.mount('/content/drive')
# project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

project_path = "../"

# Project Seed for Reproducability
SEED = random.randint(0, 2**32 - 1)  # Random integer between 0 and 2^32-1
print(f"seed: {SEED}")

model_name = "ProsusAI/finbert"

seed: 193601912


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/horizontal_roterien_katze/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/horizontal_roterien_katze/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Choose Device

In [3]:
# Detect available device
if torch.cuda.is_available():
    # check if ROCm backend is active
    if torch.version.hip is not None:
        backend = "ROCm"
    else:
        backend = "CUDA"

    device = torch.device("cuda")
    print(f"PyTorch is using GPU: {torch.cuda.get_device_name(0)}")
    print(f"Backend: {backend}")
else:
    device = torch.device("cpu")
    print("PyTorch is not using GPU — running on CPU")

PyTorch is using GPU: AMD Radeon 880M
Backend: ROCm


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## Import Data

In [4]:
before_date = "2025-11"

# Data path
categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data2.csv")

# Import Data
news_data = pd.read_csv(filepath_or_buffer=categorized_data_path, sep=',')

In [5]:
news_data.head()

,index,uuid,title,description,keywords,snippet,url,image_url,language,published_at,source,relevance_score,entities,similar,sentiment,text,clean_text,categorical_sentiment_3_class,length
0,0,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
1,1,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259
2,2,9084e5f1-75f5-4f15-aa3d-0676073b4aaf,Global week ahead: The start of a Santa Rally ...,NaN,"STOXX 600, business news",And just like that... December is upon us. It'...,https://www.cnbc.com/2025/11/30/global-week-ah...,https://image.cnbcfm.com/api/v1/image/10823257...,en,2025-11-30T05:10:58.000000Z,cnbc.com,NaN,"[{'symbol': 'M', 'name': ""Macy's, Inc."", 'exch...",[],0.6908,And just like that... December is upon us. It'...,and just like that december is upon us it is b...,positive,493
3,3,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
4,4,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259


In [6]:
news_data.isna().sum()

index                                0
uuid                                 0
title                                0
description                       3148
keywords                         37795
snippet                            216
url                                  0
image_url                          389
language                             0
published_at                         0
source                               0
relevance_score                  77088
entities                             0
similar                              0
sentiment                            4
text                                 0
clean_text                           0
categorical_sentiment_3_class        0
length                               0
dtype: int64

In [7]:
news_data = news_data.dropna(subset=["sentiment"])

In [8]:
news_data.isna().sum()

index                                0
uuid                                 0
title                                0
description                       3148
keywords                         37792
snippet                            216
url                                  0
image_url                          389
language                             0
published_at                         0
source                               0
relevance_score                  77084
entities                             0
similar                              0
sentiment                            0
text                                 0
clean_text                           0
categorical_sentiment_3_class        0
length                               0
dtype: int64

In [9]:
value_maps = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

news_data["categorical_sentiment_3_class_value_maps"] = news_data["categorical_sentiment_3_class"].map(value_maps)

## Split the data to Train, Test, and Validation

In [10]:
test_size = 0.20
val_size = 0.50

# Splitting the data into train and temp (which will be further split into validation and test)
train_df, test_df = train_test_split(news_data, test_size=test_size, random_state=SEED, stratify=news_data["categorical_sentiment_3_class_value_maps"])

# Splitting train into validation and test sets
val_df, test_df = train_test_split(test_df, test_size=val_size, random_state=SEED, stratify=test_df["categorical_sentiment_3_class_value_maps"])

In [11]:
train_df.shape, test_df.shape, val_df.shape

((61667, 20), (7709, 20), (7708, 20))

In [12]:
train_df = train_df[["clean_text", "categorical_sentiment_3_class_value_maps"]]
train_df = train_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

val_df = val_df[["clean_text", "categorical_sentiment_3_class_value_maps"]]
val_df = val_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

test_df = test_df[["clean_text","categorical_sentiment_3_class_value_maps"]]
test_df = test_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

## Data Preprocessing

### Convert Dataframe to Dataset for the Model Trainer

In [13]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

### Tokenizer for the text

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples, tokenizer=tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128   # keep this
    )

In [15]:
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

Map: 100%|██████████| 7709/7709 [00:00<00:00, 7939.69 examples/s]


In [16]:
# remove unwanted columns
train_ds = train_ds.remove_columns(["text", "__index_level_0__"])
val_ds = val_ds.remove_columns(["text", "__index_level_0__"])
test_ds = test_ds.remove_columns(["text", "__index_level_0__"])

## Train Model

### Model Arguments

In [17]:
model_checkpoint_directory = os.path.join(project_path, "model_checkpoints/finbert_categorical")
os.makedirs(model_checkpoint_directory, exist_ok=True)
learning_rate=2e-5

training_args = TrainingArguments(
    output_dir=model_checkpoint_directory,
    report_to="wandb",
    run_name=f"finbert_categorical_{learning_rate}",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    learning_rate=learning_rate,

    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    greater_is_better=True,

    save_total_limit=2,

    seed=SEED
)

/home/horizontal_roterien_katze/miniforge3/envs/thesis/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### Model Metrics for Training

In [18]:
def model_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # convert logits → probabilities
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    accuracy = accuracy_score(labels, predictions)

    try:
        roc_auc = roc_auc_score(labels, probs, multi_class='ovr')
    except:
        roc_auc = None

    return {
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "roc_auc": roc_auc
    }

In [19]:
print(train_ds.column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


### wandb setup

In [20]:
wandb.login()

# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="Thesis-Stock-Sentiment-Analysis"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="true"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/horizontal_roterien_katze/.netrc.
wandb: Currently logged in as: andreas-lukito001 (andreas-lukito001-binus-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Model Trainer

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # Since there are three classes ["Negative", "Neutral", "Positive"]
    ignore_mismatched_sizes=True
)

In [22]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    compute_metrics = model_metrics
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

In [24]:
train_ds.column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [ ]:
trainer.train()

model_save_path = os.path.join(model_checkpoint_directory, "final_model")
os.makedirs(model_save_path, exist_ok=True)
trainer.save_model(model_save_path)

  0%|          | 0/9640 [00:00<?, ?it/s]/home/horizontal_roterien_katze/miniforge3/envs/thesis/lib/python3.11/site-packages/transformers/models/bert/modeling_bert.py:435: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:379.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
  0%|          | 17/9640 [01:04<9:36:49,  3.60s/it]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7fd48a102ed0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fd4b8024590, execution_count=23 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7fd4b8026250, raw_cell="trainer.train()
wandb.finish()" transformed_cell="trainer.train()
wandb.finish()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/run/media/horizontal_roterien_katze/A/Documents/binus/Sem5/LEC_Text_Mining/1_Final_Project/Stock_Sentiment_Analysis/notebooks/04_FinBERT_categorical.ipynb#X64sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
wandb.finish()

## Model Evaluation

### Model Evaluation Function

In [ ]:
def evaluate_model(model, tokenizer, dataset, device):
    model.eval()
    
    loader = DataLoader(dataset, batch_size=32)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            preds = torch.argmax(outputs.logits, dim=1)
            labels = batch["labels"]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 4. Metrics (Standard sklearn calls)
    report = classification_report(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_preds, multi_class='ovr')
    
    roc_display = RocCurveDisplay.from_predictions(all_labels, all_preds, name="Finbert")
    
    return report, cm, roc_auc, roc_display

### Get the Model

In [27]:
trained_model = trainer.model

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7fd48a102ed0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7fd406e17f90, raw_cell="trained_model = trainer.model" transformed_cell="trained_model = trainer.model
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/run/media/horizontal_roterien_katze/A/Documents/binus/Sem5/LEC_Text_Mining/1_Final_Project/Stock_Sentiment_Analysis/notebooks/04_FinBERT_categorical.ipynb#Y104sZmlsZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7fd48a102ed0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fd406e16bd0, execution_count=27 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7fd406e17f90, raw_cell="trained_model = trainer.model" transformed_cell="trained_model = trainer.model
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/run/media/horizontal_roterien_katze/A/Documents/binus/Sem5/LEC_Text_Mining/1_Final_Project/Stock_Sentiment_Analysis/notebooks/04_FinBERT_categorical.ipynb#Y104sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
classification_report, confusion_matrix = evaluate_model(
                                        trained_model,
                                        test_ds,
                                        device
                                        )

print("========== Classification Report ==========")
print(classification_report)

print("========== Confusion Matrix ==========")
print(confusion_matrix)

print("========== ROC_AUC ==========")
print(roc_auc)

print("========== ROC Curve ==========")
roc_display.plot()

NameError: name 'test_loader' is not defined